# Notebook 02 — Deriving Observed Delay from Vehicle Location Data

**ST5011CEM Big Data Programming Project**

Notebook 01 produced the *scheduled* side: when every bus was timetabled to
reach every stop. This notebook produces the *observed* side, by matching raw
GPS pings collected from the BODS live feed to the stops they passed, and
measuring the difference.

That difference is the delay — the target variable for the models in notebook 04.

**Method.** BODS location data reports where a vehicle is, not which stop it has
reached. Arrival at a stop is therefore *inferred*: a vehicle is treated as
having served a stop at the moment of its closest approach, provided it came
within a threshold distance. The technique and its limitations are discussed in
§7 and should be repeated in the report's Critical Reflection.

## 1. Setup

In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import time
from pathlib import Path

from pyspark.sql import SparkSession, functions as F, Window
import time
from pathlib import Path

from pyspark.sql import SparkSession, functions as F, Window

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")

PROJECT   = find_project_root(Path.cwd())
AVL_DIR   = PROJECT / "data" / "raw" / "avl"
PROCESSED = PROJECT / "data" / "processed"

print("Project root :", PROJECT)
print("AVL files    :", len(list(AVL_DIR.glob("*.csv.gz"))))
print("Schedule      :", (PROCESSED / "scheduled_stops").is_dir())

Project root : E:\BODS-project
AVL files    : 23
Schedule      : True


In [2]:
spark = (SparkSession.builder
         .appName("ST5011CEM_Delay_Derivation")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.adaptive.enabled", "false")
         .config("spark.sql.session.timeZone", "Europe/London")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark UI: http://ujwal:4041


## 2. Load the collected vehicle location data

Spark reads the whole directory of hourly gzip CSVs as one DataFrame. Each file
is a separate split, so the collector's hourly rotation gives natural
parallelism at no cost.

In [3]:
t0 = time.time()
avl_raw = (spark.read
           .option("header", True)
           .csv((AVL_DIR).as_posix() + "/*.csv.gz"))

raw_count = avl_raw.count()
print(f"Raw AVL observations : {raw_count:,}")
print(f"Read in                {time.time()-t0:.1f}s")
print(f"Partitions           : {avl_raw.rdd.getNumPartitions()}")
print(f"\n100,000-record threshold met by AVL alone: {raw_count >= 100_000}")

Raw AVL observations : 3,255,969
Read in                9.5s
Partitions           : 20

100,000-record threshold met by AVL alone: True


In [4]:
print("Collection window:")
avl_raw.select(
    F.min("recorded_at_time").alias("first_observation"),
    F.max("recorded_at_time").alias("last_observation"),
    F.countDistinct("vehicle_ref").alias("distinct_vehicles"),
    F.countDistinct("operator_ref").alias("distinct_operators"),
    F.countDistinct("line_ref").alias("distinct_lines"),
).show(truncate=False)

Collection window:
+-------------------------+-------------------------+-----------------+------------------+--------------+
|first_observation        |last_observation         |distinct_vehicles|distinct_operators|distinct_lines|
+-------------------------+-------------------------+-----------------+------------------+--------------+
|2026-07-23T09:09:29+00:00|2026-07-27T07:10:54+00:00|2109             |32                |388           |
+-------------------------+-------------------------+-----------------+------------------+--------------+



## 3. Cleaning and de-duplication

The collector polls every 30 seconds, but a vehicle only reports when its
onboard system transmits. If a bus has not reported since the previous poll, the
feed returns the *same* observation again. These repeats were deliberately kept
at capture time so that the de-duplication is visible here as a pipeline step
rather than hidden in the collector.

`(vehicle_ref, recorded_at_time)` uniquely identifies a genuine observation.

In [5]:
avl = (avl_raw
       .withColumn("recorded_ts", F.to_timestamp("recorded_at_time"))
       .withColumn("lat", F.col("latitude").cast("double"))
       .withColumn("lon", F.col("longitude").cast("double"))
       .withColumn("line_norm", F.upper(F.trim(F.col("line_ref")))))

# Quality report before filtering
print("Null / invalid counts:")
avl.select(
    F.sum(F.col("recorded_ts").isNull().cast("int")).alias("bad_timestamp"),
    F.sum(F.col("lat").isNull().cast("int")).alias("bad_latitude"),
    F.sum(F.col("lon").isNull().cast("int")).alias("bad_longitude"),
    F.sum(F.col("line_norm").isNull().cast("int")).alias("missing_line"),
    F.sum(F.col("operator_ref").isNull().cast("int")).alias("missing_operator"),
).show()

avl = avl.filter(
    F.col("recorded_ts").isNotNull() &
    F.col("lat").isNotNull() & F.col("lon").isNotNull() &
    F.col("line_norm").isNotNull()
)
valid = avl.count()
print(f"Valid rows: {valid:,}  ({100*valid/raw_count:.1f}% of raw)")

Null / invalid counts:
+-------------+------------+-------------+------------+----------------+
|bad_timestamp|bad_latitude|bad_longitude|missing_line|missing_operator|
+-------------+------------+-------------+------------+----------------+
|            0|           0|            0|           0|               0|
+-------------+------------+-------------+------------+----------------+

Valid rows: 3,255,969  (100.0% of raw)


In [6]:
before = avl.count()
avl = avl.dropDuplicates(["vehicle_ref", "recorded_ts"])
after = avl.count()

print(f"Before de-duplication : {before:,}")
print(f"After  de-duplication : {after:,}")
print(f"Duplicate rate        : {100*(before-after)/before:.1f}%")
print("\nThe duplicate rate reflects how often vehicles report more slowly")
print("than the 30-second polling interval.")

avl.cache()
avl.count()

Before de-duplication : 3,255,969
After  de-duplication : 2,106,354
Duplicate rate        : 35.3%

The duplicate rate reflects how often vehicles report more slowly
than the 30-second polling interval.


2106354

## 4. Load the schedule and diagnose the join keys

This is the step most likely to need adjustment. The two feeds are published by
different systems: the GTFS timetable identifies a service by `route_short_name`
and an agency code, while SIRI-VM uses `line_ref` and a National Operator Code.
They usually agree, but not always.

The diagnostics below report the actual overlap. **If the line overlap is low,
stop and read the printed samples before continuing** — the normalisation in §3
may need changing to match your data.

In [7]:
sched = spark.read.parquet((PROCESSED / "scheduled_stops").as_posix())
sched = sched.withColumn("line_norm",
                         F.upper(F.trim(F.col("route_short_name").cast("string"))))
print(f"Scheduled stop events: {sched.count():,}")

print("\nAVL operator codes (sample):")
avl.select("operator_ref").distinct().show(10, truncate=False)

print("GTFS agencies (sample):")
sched.select("agency_id", "agency_name").distinct().show(10, truncate=False)

Scheduled stop events: 4,665,277

AVL operator codes (sample):
+------------+
|operator_ref|
+------------+
|TPEN        |
|BNFM        |
|VISB        |
|HOWT        |
|HWCO        |
|MBTO        |
|DAGC        |
|RSTY        |
|AMSY        |
|LNUD        |
+------------+
only showing top 10 rows

GTFS agencies (sample):
+---------+---------------------------+
|agency_id|agency_name                |
+---------+---------------------------+
|OP31     |D & G Bus                  |
|OP2350   |First Halifax              |
|OP268    |Kirkby Lonsdale Coaches    |
|OP5051   |FlixBus                    |
|OP326    |Pilkingtonbus              |
|OP325    |Coastliner Buses           |
|OP10     |M & H Coaches              |
|OP275    |Western Dales Community Bus|
|OP220    |Stagecoach North East      |
|OP252    |Aimee's                    |
+---------+---------------------------+
only showing top 10 rows



In [8]:
avl_lines   = set(r[0] for r in avl.select("line_norm").distinct().collect())
sched_lines = set(r[0] for r in sched.select("line_norm").distinct().collect())
overlap     = avl_lines & sched_lines

print(f"Distinct lines in AVL      : {len(avl_lines):,}")
print(f"Distinct lines in schedule : {len(sched_lines):,}")
print(f"Lines present in both      : {len(overlap):,}")
print(f"Coverage of AVL lines      : {100*len(overlap)/max(len(avl_lines),1):.1f}%")

if len(overlap) < 0.3 * len(avl_lines):
    print("\n*** LOW OVERLAP - inspect these samples before continuing ***")
    print("AVL only  :", sorted(list(avl_lines - sched_lines))[:15])
    print("Sched only:", sorted(list(sched_lines - avl_lines))[:15])
else:
    print("\nOverlap is sufficient to proceed.")

Distinct lines in AVL      : 388
Distinct lines in schedule : 939
Lines present in both      : 367
Coverage of AVL lines      : 94.6%

Overlap is sufficient to proceed.


## 5. Spatial binning

Every ping must be compared against nearby stops. A direct cross join would
produce (pings x stops) rows — with a few million pings and tens of thousands of
stops that is trillions of comparisons, which will not complete.

Instead both sides are assigned to a grid cell of roughly 330 m. Joining on the
cell reduces the problem to comparing each ping only against stops in its own
neighbourhood. Pings are replicated into the 8 surrounding cells as well, so a
stop just across a cell boundary is not missed.

This turns an O(n x m) cross join into an O(n) hash join, and is the single most
important optimisation in the pipeline.

In [9]:
CELL_DEG = 0.003          # ~330 m of latitude
MAX_DIST_M = 100          # a ping within this distance counts as serving the stop

def add_grid(df, lat_col, lon_col):
    return (df
            .withColumn("gy", F.floor(F.col(lat_col) / CELL_DEG).cast("int"))
            .withColumn("gx", F.floor(F.col(lon_col) / CELL_DEG).cast("int")))

stops = add_grid(
    sched.select("stop_id", "stop_name", "stop_lat", "stop_lon").distinct(),
    "stop_lat", "stop_lon")
print(f"Distinct stops: {stops.count():,}")

offsets = (spark.range(9)
           .withColumn("dy", (F.col("id") / 3).cast("int") - 1)
           .withColumn("dx", (F.col("id") % 3).cast("int") - 1)
           .select("dy", "dx"))

pings = (add_grid(avl, "lat", "lon")
         .crossJoin(F.broadcast(offsets))
         .withColumn("gy", F.col("gy") + F.col("dy"))
         .withColumn("gx", F.col("gx") + F.col("dx"))
         .drop("dy", "dx"))

print(f"Pings after 3x3 replication: {pings.count():,}")

Distinct stops: 35,769
Pings after 3x3 replication: 18,957,186


## 6. Haversine distance

Great-circle distance between the ping and the candidate stop. Implemented with
Spark SQL functions rather than a Python UDF: a UDF would serialise every row
between the JVM and a Python worker, whereas these expressions compile into the
query plan and run natively.

In [10]:
EARTH_R = 6371000.0   # metres

def haversine(lat1, lon1, lat2, lon2):
    phi1, phi2 = F.radians(lat1), F.radians(lat2)
    dphi       = F.radians(lat2 - lat1)
    dlambda    = F.radians(lon2 - lon1)
    a = F.sin(dphi / 2) ** 2 + F.cos(phi1) * F.cos(phi2) * F.sin(dlambda / 2) ** 2
    return F.lit(2 * EARTH_R) * F.asin(F.sqrt(a))

candidates = (pings.join(stops, ["gy", "gx"])
              .withColumn("dist_m", haversine(F.col("lat"), F.col("lon"),
                                              F.col("stop_lat"), F.col("stop_lon"))))

t0 = time.time()
n_cand = candidates.count()
print(f"Candidate ping-stop pairs : {n_cand:,}   ({time.time()-t0:.1f}s)")

near = candidates.filter(F.col("dist_m") <= MAX_DIST_M)
n_near = near.count()
print(f"Within {MAX_DIST_M} m               : {n_near:,}")
print(f"\nA full cross join would have been "
      f"{avl.count() * stops.count():,} comparisons.")

Candidate ping-stop pairs : 34,897,432   (2.5s)
Within 100 m               : 6,686,730

A full cross join would have been 75,342,176,226 comparisons.


## 7. Inferring arrival events

A bus approaching, serving and leaving a stop produces several pings inside the
threshold. Only one of them represents the arrival: the closest approach.

A window partitioned by vehicle, stop and hour selects it. The hour component
matters because a bus on a circular route may serve the same stop several times
in a shift, and each pass should be treated as a separate event.

**Limitation to record in the report:** closest approach is a proxy for arrival,
not a measurement of it. A vehicle held in traffic beside a stop it does not
serve will register a false arrival, and the 30-second polling interval bounds
the timing precision. Both are quantified in §9.

In [11]:
near = near.withColumn("hour_bucket", F.date_trunc("hour", F.col("recorded_ts")))

w_closest = Window.partitionBy("vehicle_ref", "stop_id", "hour_bucket").orderBy("dist_m")

arrivals = (near
            .withColumn("rn", F.row_number().over(w_closest))
            .filter(F.col("rn") == 1)
            .drop("rn"))

n_arr = arrivals.count()
print(f"Inferred arrival events: {n_arr:,}")

arrivals.select(
    F.round(F.mean("dist_m"), 1).alias("mean_dist_m"),
    F.round(F.expr("percentile_approx(dist_m, 0.5)"), 1).alias("median_dist_m"),
    F.round(F.max("dist_m"), 1).alias("max_dist_m"),
).show()

Inferred arrival events: 1,958,822
+-----------+-------------+----------+
|mean_dist_m|median_dist_m|max_dist_m|
+-----------+-------------+----------+
|       39.0|         32.4|     100.0|
+-----------+-------------+----------+



## 8. Matching arrivals to the timetable

Each inferred arrival is compared against the scheduled arrivals at that stop for
that line, and assigned to the nearest one in time. Delay is the signed
difference: positive means late, negative means early.

**Limitation to record in the report:** nearest-in-time assignment cannot
distinguish a bus running very late from the following bus running slightly
early. The method is therefore reliable for delays below half the service
headway and degrades beyond it. Observations beyond +/- 30 minutes are discarded
as unmatchable rather than treated as extreme delays.

In [12]:
sched_slim = sched.select(
    "stop_id", "line_norm", "arrival_sec", "arrival_hour", "next_day",
    "trip_id", "route_id", "direction_id", "stop_sequence",
    "agency_id", "agency_name", "route_short_name", "stop_lat", "stop_lon")

arr = (arrivals
       .withColumn("obs_sec",
                   F.hour("recorded_ts") * 3600 +
                   F.minute("recorded_ts") * 60 +
                   F.second("recorded_ts"))
       .select("vehicle_ref", "stop_id", "line_norm", "recorded_ts", "obs_sec",
               "dist_m", "operator_ref", "direction_ref", "lat", "lon"))

joined = (arr.join(sched_slim, ["stop_id", "line_norm"])
             .withColumn("delay_sec", F.col("obs_sec") - F.col("arrival_sec")))

w_best = Window.partitionBy("vehicle_ref", "stop_id", "recorded_ts") \
               .orderBy(F.abs(F.col("delay_sec")))

matched = (joined
           .withColumn("rn", F.row_number().over(w_best))
           .filter(F.col("rn") == 1)
           .drop("rn")
           .filter(F.abs(F.col("delay_sec")) <= 30 * 60))

matched.cache()
n_matched = matched.count()
print(f"Matched delay observations: {n_matched:,}")
print(f"Match rate from arrivals  : {100*n_matched/max(n_arr,1):.1f}%")

Matched delay observations: 1,201,509
Match rate from arrivals  : 61.3%


In [13]:
print("Delay distribution (minutes):")
matched.select(
    F.round(F.mean("delay_sec") / 60, 2).alias("mean"),
    F.round(F.expr("percentile_approx(delay_sec, 0.5)") / 60, 2).alias("median"),
    F.round(F.stddev("delay_sec") / 60, 2).alias("std_dev"),
    F.round(F.expr("percentile_approx(delay_sec, 0.05)") / 60, 2).alias("p05"),
    F.round(F.expr("percentile_approx(delay_sec, 0.95)") / 60, 2).alias("p95"),
    F.round(F.skewness("delay_sec"), 3).alias("skewness"),
    F.round(F.kurtosis("delay_sec"), 3).alias("kurtosis"),
).show()

print("Sample of matched observations:")
matched.select("vehicle_ref", "route_short_name", "stop_name" if "stop_name" in matched.columns else "stop_id",
               "recorded_ts", "obs_sec", "arrival_sec",
               F.round(F.col("delay_sec") / 60, 2).alias("delay_min"),
               F.round("dist_m", 1).alias("dist_m")).show(10, truncate=False)

Delay distribution (minutes):
+----+------+-------+-----+----+--------+--------+
|mean|median|std_dev|  p05| p95|skewness|kurtosis|
+----+------+-------+-----+----+--------+--------+
| 0.5|  0.35|   5.55|-7.23|8.85|  -0.032|   7.448|
+----+------+-------+-----+----+--------+--------+

Sample of matched observations:
+-----------+----------------+-----------+-------------------+-------+-----------+---------+------+
|vehicle_ref|route_short_name|stop_id    |recorded_ts        |obs_sec|arrival_sec|delay_min|dist_m|
+-----------+----------------+-----------+-------------------+-------+-----------+---------+------+
|01T_E5T    |163             |1800MNBS0M1|2026-07-24 16:25:05|59105  |59160      |-0.92    |15.2  |
|01T_E5T    |163             |1800NB04271|2026-07-24 17:22:35|62555  |62640      |-1.42    |42.8  |
|01T_E5T    |163             |1800NB04281|2026-07-24 17:19:31|62371  |62580      |-3.48    |8.2   |
|01T_E5T    |163             |1800NB08561|2026-07-24 16:44:02|60242  |60300      |

## 9. Reliability metrics from the assignment brief

The brief defines Service Reliability as the proportion of trips arriving within
+/- 2 minutes of the timetable in urban areas, with a target of 85%. Travel Time
Variability is the coefficient of variation of trip duration, target 15% or
below.

These are computed here so the report can state them against the stated
thresholds rather than inventing new measures.

In [14]:
matched = matched.withColumn("on_time", (F.abs(F.col("delay_sec")) <= 120).cast("int"))
matched.createOrReplaceTempView("delays")

print("Overall Service Reliability:")
spark.sql("""
    SELECT COUNT(*)                                        AS observations,
           ROUND(100.0 * SUM(on_time) / COUNT(*), 2)       AS on_time_pct,
           ROUND(AVG(delay_sec) / 60.0, 2)                 AS mean_delay_min,
           CASE WHEN 100.0 * SUM(on_time) / COUNT(*) >= 85
                THEN 'MEETS 85% TARGET' ELSE 'BELOW TARGET' END AS verdict
    FROM delays
""").show(truncate=False)

Overall Service Reliability:
+------------+-----------+--------------+------------+
|observations|on_time_pct|mean_delay_min|verdict     |
+------------+-----------+--------------+------------+
|1201509     |54.15      |0.5           |BELOW TARGET|
+------------+-----------+--------------+------------+



In [15]:
print("Operator compliance benchmark (the stakeholder view):")
spark.sql("""
    SELECT agency_name                                      AS operator,
           COUNT(*)                                         AS observations,
           ROUND(100.0 * SUM(on_time) / COUNT(*), 2)        AS reliability_pct,
           ROUND(AVG(delay_sec) / 60.0, 2)                  AS mean_delay_min,
           ROUND(STDDEV(delay_sec) / 60.0, 2)               AS delay_sd_min
    FROM delays
    GROUP BY agency_name
    HAVING COUNT(*) >= 100
    ORDER BY reliability_pct ASC
""").show(25, truncate=False)

Operator compliance benchmark (the stakeholder view):
+------------------------------------------+------------+---------------+--------------+------------+
|operator                                  |observations|reliability_pct|mean_delay_min|delay_sd_min|
+------------------------------------------+------------+---------------+--------------+------------+
|National Express                          |254         |22.05          |0.02          |10.76       |
|Preston Bus                               |3592        |27.00          |1.52          |13.81       |
|Vision Bus                                |892         |27.80          |1.02          |6.82        |
|Holmeswood Coaches Ltd                    |3191        |31.81          |1.46          |11.82       |
|The Burnley Bus Company                   |5878        |32.36          |1.38          |9.06        |
|Huyton Travel                             |6145        |33.64          |1.79          |10.8        |
|Link Network               

In [16]:
print("Travel Time Variability - coefficient of variation by route:")
spark.sql("""
    SELECT route_short_name                                  AS route,
           COUNT(*)                                          AS observations,
           ROUND(AVG(delay_sec) / 60.0, 2)                   AS mean_delay_min,
           ROUND(STDDEV(delay_sec) / NULLIF(ABS(AVG(delay_sec)), 0), 3) AS cv
    FROM delays
    GROUP BY route_short_name
    HAVING COUNT(*) >= 50
    ORDER BY cv DESC
    LIMIT 20
""").show(truncate=False)

Travel Time Variability - coefficient of variation by route:
+-----+------------+--------------+--------+
|route|observations|mean_delay_min|cv      |
+-----+------------+--------------+--------+
|442  |1006        |0.01          |1918.018|
|65   |376         |-0.02         |606.126 |
|180  |717         |-0.02         |550.088 |
|50   |12260       |0.0           |410.513 |
|594  |1289        |0.04          |288.095 |
|118  |7075        |-0.02         |180.984 |
|375  |2305        |-0.04         |168.595 |
|192  |38092       |0.01          |129.302 |
|287  |178         |0.08          |104.373 |
|31   |938         |-0.03         |103.383 |
|391  |2570        |-0.12         |90.826  |
|469  |1054        |0.04          |85.521  |
|119  |3390        |0.07          |81.485  |
|43   |15131       |-0.03         |79.599  |
|461  |728         |-0.12         |75.582  |
|216  |8340        |0.03          |66.755  |
|3A   |612         |0.2           |65.186  |
|613  |607         |0.14          |64.3

In [17]:
print("Delay by hour of day - the peak effect the models must learn:")
spark.sql("""
    SELECT HOUR(recorded_ts)                          AS hour,
           COUNT(*)                                   AS observations,
           ROUND(AVG(delay_sec) / 60.0, 2)            AS mean_delay_min,
           ROUND(100.0 * SUM(on_time) / COUNT(*), 1)  AS on_time_pct
    FROM delays
    GROUP BY HOUR(recorded_ts)
    ORDER BY hour
""").show(24, truncate=False)

Delay by hour of day - the peak effect the models must learn:
+----+------------+--------------+-----------+
|hour|observations|mean_delay_min|on_time_pct|
+----+------------+--------------+-----------+
|0   |5           |-9.56         |20.0       |
|1   |2           |1.21          |0.0        |
|3   |1           |8.95          |0.0        |
|4   |228         |-5.07         |51.8       |
|5   |4374        |-0.72         |53.0       |
|6   |33          |0.18          |45.5       |
|7   |31259       |-0.18         |55.9       |
|8   |16512       |-0.1          |56.5       |
|9   |310         |-0.25         |56.8       |
|10  |76502       |0.49          |54.7       |
|11  |125464      |0.64          |52.6       |
|12  |78698       |0.9           |48.6       |
|13  |79736       |0.95          |47.3       |
|14  |135820      |0.78          |50.4       |
|15  |149017      |0.39          |55.2       |
|16  |154139      |0.33          |57.0       |
|17  |153533      |0.26          |57.6       

## 10. Persist

Written to Parquet partitioned by operator. Notebook 03 (EDA) and notebook 04
(models) both read this file rather than repeating the matching, which is by far
the most expensive stage of the pipeline.

In [18]:
OUT = (PROCESSED / "observed_delays").as_posix()

t0 = time.time()
(matched
 .withColumn("delay_min", F.col("delay_sec") / 60.0)
 .withColumn("obs_date", F.to_date("recorded_ts"))
 .write.mode("overwrite").partitionBy("agency_id").parquet(OUT))

print(f"Written in {time.time()-t0:.1f}s")

check = spark.read.parquet(OUT)
print(f"Rows        : {check.count():,}")
print(f"Columns     : {len(check.columns)}")
print(f"Partitions  : {check.rdd.getNumPartitions()}")
check.printSchema()

Written in 5.0s
Rows        : 1,201,509
Columns     : 26
Partitions  : 20
root
 |-- stop_id: string (nullable = true)
 |-- line_norm: string (nullable = true)
 |-- vehicle_ref: string (nullable = true)
 |-- recorded_ts: timestamp (nullable = true)
 |-- obs_sec: integer (nullable = true)
 |-- dist_m: double (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- direction_ref: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- arrival_sec: integer (nullable = true)
 |-- arrival_hour: integer (nullable = true)
 |-- next_day: integer (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- route_id: integer (nullable = true)
 |-- direction_id: integer (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- route_short_name: string (nullable = true)
 |-- stop_lat: double (nullable = true)
 |-- stop_lon: double (nullable = true)
 |-- delay_sec: integer (nullable = true)
 

## 11. Screenshots

Before stopping the session, capture for the report:

* Spark UI **Stages** tab — the grid join stage, showing task parallelism
* Spark UI **SQL / DataFrame** tab — the query plan for the spatial join
* The §9 operator compliance table (this is the stakeholder-facing result)
* The §8 delay distribution statistics

In [19]:
print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Capture screenshots now, then run the final cell.")

Spark UI: http://ujwal:4041
Capture screenshots now, then run the final cell.


In [ ]:
spark.stop()
print("Notebook 02 complete.")